# Numerical Linear Algebra

[![Matrix transform](fig/matrix_transform.png)](https://xkcd.com/184/)

## Why Should an Astronomer Care?

Linear algebra is a fundamental part of modern mathematics. It supports fields from calculus and statistics to geometry, control theory, and functional analysis. Most linear systems are well understood. Even nonlinear problems are often studied through linear approximations.

Numerical linear algebra extends these ideas to computation, enabling solutions of PDEs, optimization tasks, eigenvalue problems, and more. It addresses some of the hardest problems in physics and engineering.

A few examples from astronomy:

| You want to... | Inside the code, you are... |
|---|---|
| Fit a model (polynomial spectrum, light curve, ...) | Solving a least-squares system $A\mathbf{c} \approx \mathbf{y}$ |
| Compute the gravitational potential on a grid | Solving Poisson's equation $\nabla^2\Phi = 4\pi G\rho$, a huge sparse system |
| Evolve stiff chemistry or radiative transfer implicitly | Solving $A\mathbf{x}=\mathbf{b}$ at every time step |
| Reconstruct an image from radio interferometer data | Inverting a (very large) linear measurement operator |
| Infer a star's interior from its oscillation frequencies (asteroseismology) | Solving an eigenvalue problem for normal modes |
| Compress thousands of galaxy spectra into a few templates | Computing an SVD / PCA |


### Gravity on a Grid

In a simulation with self-gravity.
At every time step the code must find the gravitational potential
$\Phi$ from the current density $\rho$ by solving Poisson's equation
\begin{align}
  \nabla^2\Phi = 4\pi G\rho.
\end{align}
On a grid with spacing $h$, the Laplacian at each cell is approximated
by its six neighbors:
\begin{align}
  \frac{\Phi_{i+1,j,k} + \Phi_{i-1,j,k} + \Phi_{i,j+1,k} + \Phi_{i,j-1,k} + \Phi_{i,j,k+1} + \Phi_{i,j,k-1} - 6\Phi_{i,j,k}}{h^2}
  = 4\pi G\rho_{i,j,k}.
\end{align}
That is one linear equation per cell.
A modest $256^3$ grid has about $1.7\times10^7$ unknowns, but each
equation involves only 7 of them: the matrix is huge but extremely
**sparse**.
It is the 3D version of the spring chain problem we study below.

Storing this matrix as a full array, or solving it with plain Gaussian
elimination, is hopeless.
Simulation codes use fast Fourier transforms or **multigrid** iterative
methods instead, which we preview in the section on sparse matrices and
iterative solvers.

### Imaging with Radio Interferometers

An interferometer such as the VLA, ALMA, or the Event Horizon
Telescope does not take a picture directly.
Each pair of antennas (a baseline) measures one Fourier component of
the sky brightness $I(l, m)$, the **visibility**
\begin{align}
  V(u, v) = \iint I(l, m)\, e^{-2\pi i (ul + vm)}\, dl\, dm,
\end{align}
where $(u, v)$ is the baseline length and orientation measured in
wavelengths.
Discretizing the image into pixels turns this into
\begin{align}
  \mathbf{V} = F\,\mathbf{I} + \text{noise},
\end{align}
where $F$ is a (partial) Fourier matrix with one row per measured
visibility and one column per pixel.

Reconstructing the image means inverting $F$, but the antennas sample
only some of the $(u, v)$ plane.
There are fewer independent measurements than pixels, so infinitely many
images fit the data exactly, and $F$ is very ill-conditioned.
For a $1000\times1000$ pixel image, $F$ would have $10^6$ columns and
millions of rows.

### Compressing Spectra with PCA

A large spectroscopic survey (SDSS, DESI, ...) collects millions of
spectra, each with thousands of wavelength pixels.
Stack $N$ spectra as the rows of a data matrix $X$ ($N\times M$, with $M$
pixels).
Galaxy spectra are not random: they are built from a limited number of
physical ingredients (old stars, young stars, gas emission lines,
dust), so the rows of $X$ are highly redundant.

After subtracting the mean spectrum $\bar{\mathbf{s}}$ from every row,
the singular value decomposition finds the
orthogonal "**eigenspectra**" that capture the most
variance.
A typical galaxy spectrum can then be written as
\begin{align}
  \mathbf{s} \approx \bar{\mathbf{s}} + a_1\mathbf{e}_1 + a_2\mathbf{e}_2 + \dots + a_k\mathbf{e}_k,
\end{align}
with only $k \sim 5$-$10$ coefficients instead of thousands of pixel
values.
These coefficients are useful for classifying galaxies, flagging
unusual objects, and filling in missing or noisy pixels.

### What You Need to Know

You will almost never write these solvers yourself: `numpy` and
`scipy` call highly optimized libraries (LAPACK, BLAS) for you.
But we want to know
1. which tool to call for which problem,
2. how expensive it is, and
3. when we can trust the answer.

### Plan: One Physical System, Many Questions

Instead of jumping between unrelated random matrices, we follow
**one simple physical system** throughout the lecture: a chain of
$N$ identical masses connected by springs between two walls.
```
|--k--[m]--k--[m]--k-- ... --k--[m]--k--|
```
Each question we ask about it leads naturally to one topic:

| Question about the chain | Topic |
|---|---|
| If I push on the masses, where do they settle? | Solving $A\mathbf{x}=\mathbf{b}$ (direct methods) |
| How much should I trust the computer's answer? | Floating point and the condition number |
| What if there are millions of masses? | Sparse matrices and iterative solvers |
| If I let go, how does it vibrate? | Eigenvalues and eigenvectors (normal modes) |

As we will see, the same matrix also describes gravity on a grid and
standing waves in a star, so the chain is less of a toy than it
looks.

In [ ]:
import numpy as np
import scipy.linalg as la
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from matplotlib import pyplot as plt
from time import perf_counter

np.set_printoptions(precision=4, suppress=True)

## Background:

### Matrices Are Machines That Transform Vectors

A matrix $A$ acting on a vector $\mathbf{x}$ can be read in two
equivalent ways:

* **Row view:** each entry of $A\mathbf{x}$ is a dot product of a
  row of $A$ with $\mathbf{x}$.
  This is the "system of equations" view: row $i$ is equation $i$.
* **Column view:** $A\mathbf{x}$ is a weighted sum of the columns of
  $A$, with weights given by the entries of $\mathbf{x}$:
  \begin{align}
    A\mathbf{x} = x_1 \mathbf{a}_1 + x_2 \mathbf{a}_2 + \dots + x_n \mathbf{a}_n.
  \end{align}
  Solving $A\mathbf{x}=\mathbf{b}$ therefore means: *find the recipe
  (weights) that builds $\mathbf{b}$ out of the columns of $A$.*


In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
x = np.array([1.0, 2.0])

print(A @ x, "matrix-vector product A @ x")
print(x[0]*A[:,0] + x[1]*A[:,1], "same thing, as a sum of columns")
print(A * x, "NOT a matrix product: element-wise, broadcast over rows")

### Computers Keep About 16 Significant Digits

Standard (double precision) floating-point numbers store about 16
significant decimal digits.
Every arithmetic operation rounds its result to the nearest
representable number, introducing a tiny *relative* error of up to
the **machine epsilon** $\epsilon \approx 2.2\times10^{-16}$.

Usually this is harmless.
The danger is when an algorithm *amplifies* these tiny errors,
for example by subtracting two nearly equal numbers or dividing by a
very small one.
Most of the "numerical" in numerical linear algebra is about
controlling this amplification.

### Cost: Big-O Notation

We describe how the run time of an algorithm grows with the problem
size $n$ using "big-O" notation.
An $\mathcal{O}(n^3)$ algorithm takes roughly $8\times$ longer when
$n$ doubles, and $1000\times$ longer when $n$ grows by $10\times$.

For an $n\times n$ matrix:
* storing it takes $n^2$ numbers,
* multiplying it by a vector takes $\mathcal{O}(n^2)$ operations,
* solving $A\mathbf{x}=\mathbf{b}$ by elimination takes
  $\mathcal{O}(n^3)$ operations: stops working for
large simulations.

## The Running Example: Masses on Springs

Consider $N$ equal masses connected by $N+1$ identical springs (spring
constant $k$), with the two outer springs attached to fixed walls.
Let $x_i$ be the displacement of mass $i$ from its resting position,
and suppose we apply a constant external force $f_i$ to each mass.

Hooke's law says the spring between masses $i-1$ and $i$ pulls on
mass $i$ with force $k(x_{i-1} - x_i)$.
When the chain comes to rest, the forces on each mass balance:
\begin{align}
  k(x_{i-1} - x_i) + k(x_{i+1} - x_i) + f_i = 0
  \quad\Longrightarrow\quad
  -k x_{i-1} + 2k x_i - k x_{i+1} = f_i,
\end{align}
with $x_0 = x_{N+1} = 0$ at the walls.

Writing all $N$ equations at once gives a linear system
$K\mathbf{x} = \mathbf{f}$ with the **stiffness matrix**
\begin{align}
  K = k\begin{pmatrix}
     2 & -1 &        &        &    \\
    -1 &  2 & -1     &        &    \\
       & \ddots & \ddots & \ddots &    \\
       &        & -1     &  2     & -1 \\
       &        &        & -1     &  2
  \end{pmatrix}.
\end{align}
Each mass only talks to its two neighbors, so $K$ is highly sparse:
almost all of its entries are zero.
Remember this; it will matter a lot when $N$ gets large.

In [ ]:
def spring_matrix(N, k=1.0):
    '''Stiffness matrix for N equal masses joined by N+1 springs between two walls.'''
    return k * (2*np.eye(N) - np.eye(N, k=1) - np.eye(N, k=-1))

print(spring_matrix(5))

Let's ask the computer where the masses settle for two different
force patterns.

In [ ]:
N = 10
K = spring_matrix(N)

f_push    = np.zeros(N); f_push[2] = 1.0   # push mass 3 to the right
f_uniform = 0.1 * np.ones(N)               # push every mass a little to the right

x_push    = np.linalg.solve(K, f_push)
x_uniform = np.linalg.solve(K, f_uniform)

i = np.arange(1, N+1)
plt.plot(i, x_push,    'o-', label='push on mass 3')
plt.plot(i, x_uniform, 's-', label='uniform push')
plt.xlabel('mass index $i$')
plt.ylabel('displacement $x_i$')
plt.legend()

```{admonition} Astronomy connection: this is also gravity on a grid
:class: note
The combination $x_{i-1} - 2x_i + x_{i+1}$ is the finite-difference
approximation of a second derivative, $h^2\, d^2x/ds^2$, on a grid
with spacing $h$.
So $K$ is (minus) the **discrete Laplacian**, and solving
$K\mathbf{x}=\mathbf{f}$ is the same problem as solving Poisson's
equation
\begin{align}
  \frac{d^2\Phi}{ds^2} = 4\pi G\rho
  \quad\Longrightarrow\quad
  K\boldsymbol{\Phi} = -4\pi G h^2 \boldsymbol{\rho}.
\end{align}
In 3D the matrix has the same structure, just with more neighbors.
Self-gravitating simulations of disks, star clusters, and galaxies
solve systems like this at every time step.
```

In [ ]:
# 1D gravitational potential of a Gaussian density blob (units with G = 1).
# For simplicity we set Phi = 0 at both ends of the box.
N   = 200
s   = np.linspace(0, 1, N+2)[1:-1]   # interior grid points
h   = s[1] - s[0]
rho = np.exp(-0.5 * ((s - 0.5) / 0.05)**2)

Phi = np.linalg.solve(spring_matrix(N), -4*np.pi * h**2 * rho)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(s, rho); ax[0].set_xlabel('$s$'); ax[0].set_ylabel(r'density $\rho$')
ax[1].plot(s, Phi); ax[1].set_xlabel('$s$'); ax[1].set_ylabel(r'potential $\Phi$')
fig.tight_layout()

`np.linalg.solve()` gave us answers instantly.
The rest of the lecture opens this black box:
* What does `solve()` actually do, and can it fail?
* How do we know if the answer is accurate?
* What happens when $N$ is a million instead of ten?

## A Picture to Keep in Mind: Matrices Stretch Space

Before looking at algorithms, it helps to *see* what a matrix does.
Take every vector on the unit circle and multiply it by a $2\times2$
matrix $A$.
The circle always becomes an **ellipse**: $A$ stretches space by
different amounts in different directions.

Two facts about this ellipse will come back later:
* How *squashed* the ellipse is tells us how hard it is to undo $A$,
  that is, to solve $A\mathbf{x}=\mathbf{b}$ (the **condition number**).
* For a symmetric matrix, the ellipse's axes point along special
  directions that $A$ only stretches without rotating (the
  **eigenvectors**).

In [ ]:
def show_transform(A, ax, title=''):
    '''Plot the unit circle and its image under the 2x2 matrix A.'''
    theta  = np.linspace(0, 2*np.pi, 200)
    circle = np.array([np.cos(theta), np.sin(theta)])
    image  = A @ circle

    ax.plot(*circle, 'k--', lw=1, label='unit circle')
    ax.plot(*image,  'C0',  lw=2, label='after $A$')
    for e, c in [(np.array([1, 0]), 'C1'), (np.array([0, 1]), 'C2')]:
        Ae = A @ e
        ax.arrow(0, 0, *Ae, color=c, width=0.03, length_includes_head=True)
    ax.set_aspect('equal')
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
show_transform(np.array([[2.0, 0.0], [0.0, 1.0]]), axes[0], 'stretch along $x$')
show_transform(np.array([[2.0, 1.0], [1.0, 2.0]]), axes[1], 'symmetric: tilted ellipse')
show_transform(np.array([[1.0, 1.0], [1.0, 1.02]]), axes[2], 'nearly flat: hard to undo')
axes[0].legend(loc='lower left')
fig.tight_layout()

The arrows show where the unit vectors $(1,0)$ and $(0,1)$ end up.
In the left panel, the matrix is diagonal. It only stretches
the unit vectors. The resulting vectors are the columns of $A$.
In the right panel the two columns point in almost the same
direction, and the circle is crushed into a thin sliver.
Many different inputs land almost on top of each other, so recovering
the input from the output is delicate.

## Solving $A\mathbf{x}=\mathbf{b}$ Directly

### Gaussian Elimination: High-School Algebra, Automated

You already know how to solve
\begin{align}
  \begin{cases}
    2x + y = 5 \\
    4x + 3y = 13
  \end{cases}
\end{align}
by hand: subtract $2\times$ the first equation from the second to
eliminate $x$, giving $y = 3$, then substitute back to find $x = 1$.

Gaussian elimination is exactly this, done systematically:
1. **Forward elimination:**
   for each column $k$, use row $k$ (the **pivot** row) to create
   zeros below the diagonal, by subtracting
   $(a_{ik}/a_{kk})\times$ row $k$ from each row $i > k$.
   The matrix becomes upper triangular.
2. **Back substitution:**
   the last equation now has one unknown; solve it, plug into the
   equation above, and work upward.

This costs about $\frac{2}{3}n^3$ operations, the $\mathcal{O}(n^3)$
we mentioned earlier.

Here is a direct (deliberately simple, loop-based) translation into
Python.
Normally we avoid `for` loops in Python for speed, but here they make
the algorithm easy to read.

In [ ]:
def solve_Gaussian(A, b):
    '''
    Perform naive (no pivoting) Gaussian elimination to solve the
    matrix equation A x = b.
    Returns the solution vector x.
    '''
    assert A.ndim == 2 and A.shape[0] == A.shape[1]  # must be square matrix
    assert b.ndim == 1 and b.shape[0] == A.shape[1]  # must be a vector

    A = A.astype(float)  # ensure floating-point, create copy by default
    b = b.astype(float)  # ensure floating-point, create copy by default
    n = b.shape[0]

    # Forward elimination
    for k in range(n-1):
        for i in range(k+1, n):
            if A[k, k] == 0:
                raise ValueError("Zero pivot encountered (no pivoting).")
            factor    = A[i, k] / A[k, k]
            A[i, k:] -= factor * A[k, k:]
            b[i]     -= factor * b[k]

    # Back-substitution
    x = np.zeros(n)
    for i in reversed(range(n)):
        s = b[i]
        for j in range(i+1, n):
            s -= A[i, j] * x[j]
        x[i] = s / A[i, i]

    return x

Check it on our spring chain:

In [ ]:
K = spring_matrix(10)
print(solve_Gaussian(K, f_push), "naive Gaussian elimination")
print(np.linalg.solve(K, f_push), "numpy")

### When Elimination Breaks: Tiny Pivots

Each elimination step divides by the pivot $a_{kk}$.
Dividing by a very small number produces very large numbers, and the
rounding errors that come with them can wipe out the answer.

Here is the smallest possible example.
With $\varepsilon = 10^{-17}$, solve
\begin{align}
  \begin{pmatrix} \varepsilon & 1 \\ 1 & 1 \end{pmatrix}
  \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}
  =
  \begin{pmatrix} 1 \\ 2 \end{pmatrix}.
\end{align}
The exact solution is $x_1 = 1/(1-\varepsilon) \approx 1$ and
$x_2 = (1-2\varepsilon)/(1-\varepsilon) \approx 1$.

In [ ]:
eps = 1e-17
A = np.array([[eps, 1.0],
              [1.0, 1.0]])
b = np.array([1.0, 2.0])

print(solve_Gaussian(A, b), "naive Gaussian elimination")
print(np.linalg.solve(A, b), "numpy")

The naive method gets $x_1 = 0$ instead of $1$: a 100% error.
What happened?
Eliminating $x_1$ multiplies row 1 by $1/\varepsilon = 10^{17}$, so the
second row becomes
$(1 - 10^{17})\,x_2 = 2 - 10^{17}$.
In floating point, both $1 - 10^{17}$ and $2 - 10^{17}$ round to
$-10^{17}$; the information in the "1" and "2" is lost.

**The fix is simple: swap the rows.**
If we use the equation with the *largest* coefficient in the current
column as the pivot row, we never divide by a tiny number.
This is called **partial pivoting**, and every serious solver
(including `np.linalg.solve()`) does it.

In [ ]:
# HANDSON: improve the naive Gaussian elimination by adding partial pivoting.

def solve_Gaussian_pivot(A, b):
    '''
    Perform Gaussian elimination with partial pivoting to solve
    the matrix equation A x = b.
    Returns the solution vector x.
    '''
    assert A.ndim == 2 and A.shape[0] == A.shape[1]  # must be square matrix
    assert b.ndim == 1 and b.shape[0] == A.shape[1]  # must be a vector

    A = A.astype(float)  # ensure floating-point, create copy by default
    b = b.astype(float)  # ensure floating-point, create copy by default
    n = b.shape[0]

    # Forward elimination
    for k in range(n-1):
        # TODO: pivoting: find the row p >= k with the largest |A[p, k]|
        #       (hint: np.argmax)

        # TODO: swap rows k and p of both A and b

        for i in range(k+1, n):
            factor    = A[i, k] / A[k, k]
            A[i, k:] -= factor * A[k, k:]
            b[i]     -= factor * b[k]

    # Back-substitution
    x = np.zeros(n)
    for i in reversed(range(n)):
        s = b[i]
        for j in range(i+1, n):
            s -= A[i, j] * x[j]
        x[i] = s / A[i, i]

    return x

In [ ]:
# HANDSON: test your pivoting solver on the 2x2 example above,
#          and on a larger random matrix with A[0,0] set to 1e-17.

print(solve_Gaussian_pivot(A, b), "with pivoting")
print(np.linalg.solve(A, b), "numpy")

```{admonition} Key point: blame the algorithm, not the problem
:class: important
The $2\times2$ matrix above is perfectly reasonable: its rows are
nowhere near parallel, and a good algorithm solves it to full
precision.
The naive method failed because of *how* it did the arithmetic.
An algorithm that does not amplify rounding errors is called
**numerically stable**.
In the next section we meet the opposite situation: problems that
are hard *no matter which algorithm you use*.
```

### $LU$ Decomposition: Do the Expensive Work Once

Suppose you need to solve $K\mathbf{x}=\mathbf{f}$ for *many*
different force patterns $\mathbf{f}$ but the same springs $K$.
In astronomy this is common: e.g., the grid (and hence the Poisson
matrix) stays fixed while the density changes every time step.

Notice that the row operations in elimination depend only on $A$,
not on $\mathbf{b}$.
So we can record them once.
Gaussian elimination secretly factors the matrix as
\begin{align}
  A = P L U,
\end{align}
where
* $U$ is the upper-triangular matrix left after elimination,
* $L$ is lower triangular and stores the multipliers (`factor` in
  the code above),
* $P$ is a permutation matrix that records the row swaps from pivoting.

Once $A$ is factored as $P L U$, solving $A\mathbf{x} = \mathbf{b}$
becomes:
1. $L \mathbf{y} = P^T \mathbf{b}$ (forward substitution)
2. $U \mathbf{x} =     \mathbf{y}$ (back-substitution)

Gaussian elimination essentially constructs the $L$ and $U$ matrices
behind the scenes:
* The multipliers used in the row operations become the entries of
  $L$.
* The final upper-triangular form is $U$.

Factoring costs $\mathcal{O}(n^3)$ once.
After that, each new right-hand side only needs a forward and a back
substitution, which cost $\mathcal{O}(n^2)$.

In [ ]:
P, L, U = la.lu(spring_matrix(4))
print("P =\n", P)
print("L =\n", L)
print("U =\n", U)
print("P L U - K =\n", P @ L @ U - spring_matrix(4))

In practice, use `scipy.linalg.lu_factor()` to factor once and
`scipy.linalg.lu_solve()` for each right-hand side:

In [ ]:
N    = 1000
K    = spring_matrix(N)
nrhs = 100
F    = np.random.random((nrhs, N))   # 100 different force patterns

t0 = perf_counter()
for f in F:
    x = np.linalg.solve(K, f)        # re-factors K every time
t1 = perf_counter()

lu_piv = la.lu_factor(K)             # factor once
for f in F:
    x = la.lu_solve(lu_piv, f)       # cheap triangular solves
t2 = perf_counter()

print(f"solve() every time: {t1-t0:.3f} sec")
print(f"lu_factor + lu_solve: {t2-t1:.3f} sec")

## Matrix Inverse

Although the matrix inverse $A^{-1}$ is a central theoretical concept,
explicitly forming $A^{-1}$ just to solve $A \mathbf{x} = \mathbf{b}$
is almost always unnecessary and can degrade numerical stability.
Instead, direct approaches (like Gaussian Elimination or LU
factorization) find $\mathbf{x}$ with fewer operations and less error
accumulation.
Both methods cost about $\mathcal{O}(n^3)$, but computing the entire
inverse introduces extra steps and can magnify floating-point errors.

## When Can We Trust the Answer? The Condition Number

Pivoting fixes *algorithmic* problems.
But some problems are intrinsically sensitive: tiny changes in the
input cause large changes in the answer, and no algorithm can fix
that.

### A Picture: Nearly Parallel Lines

A $2\times2$ system is the intersection of two lines.
If the lines cross at a healthy angle, nudging one line moves the
intersection only a little.
If the lines are nearly parallel, the same nudge sends the
intersection far away.

In [ ]:
def plot_lines(A, b, b_nudged, ax, title=''):
    '''Plot the two lines of a 2x2 system, before and after nudging b.'''
    s = np.linspace(-1, 3, 2)
    for row, c in zip(range(2), ['C0', 'C1']):
        a1, a2 = A[row]
        ax.plot(s, (b[row]        - a1*s) / a2, c,        label=f'equation {row+1}')
        ax.plot(s, (b_nudged[row] - a1*s) / a2, c+'--')
    x  = np.linalg.solve(A, b)
    xn = np.linalg.solve(A, b_nudged)
    ax.plot(*x,  'ko', ms=8, label='solution')
    ax.plot(*xn, 'rx', ms=10, mew=3, label='after nudge')
    ax.set_xlim(-1, 3); ax.set_ylim(-1, 3); ax.set_aspect('equal')
    ax.set_title(title)
    print(f"{title}: x = {x}  ->  x_nudged = {xn}")

nudge = np.array([0.0, 0.1])  # change b2 by 0.1

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

A_good = np.array([[1.0,  1.0], [1.0, -1.0]])
b_good = np.array([2.0, 0.0])
plot_lines(A_good, b_good, b_good + nudge, axes[0], 'well-conditioned')

A_bad  = np.array([[1.0,  1.0], [1.0, 1.1]])
b_bad  = np.array([2.0, 2.1])
plot_lines(A_bad,  b_bad,  b_bad  + nudge, axes[1], 'ill-conditioned')

axes[0].legend(loc='upper right')
fig.tight_layout()

Both systems have the same solution $(1, 1)$, and in both cases we
changed $b_2$ by only $0.1$.
For the ill-conditioned system the answer jumped to $(0, 2)$.

In real life, $\mathbf{b}$ always carries some error: measurement
noise in your data, or at the very least floating-point rounding at
the level of $10^{-16}$.
An ill-conditioned matrix amplifies that error.

### Measuring Sensitivity: the Condition Number

```{admonition} Background: norms
:class: tip
The **norm** $\|\mathbf{x}\|$ of a vector measures its size; the
usual choice is the Euclidean length
$\|\mathbf{x}\|_2 = \sqrt{x_1^2 + \dots + x_n^2}$.
The norm of a matrix, $\|A\|_2$, is the largest factor by which $A$
can stretch any vector, i.e., the long semi-axis of the ellipse we
drew earlier.
```

Recall the circle-to-ellipse picture.
Let $\sigma_{\max}$ and $\sigma_{\min}$ be the longest and shortest
semi-axes of the ellipse (they are called the **singular values** of
$A$).
The **condition number** is their ratio:
\begin{align}
  \kappa(A) = \frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}
            = \|A\|_2\,\|A^{-1}\|_2 \ \geq 1.
\end{align}
A round ellipse ($\kappa \approx 1$) is easy to undo.
A thin sliver ($\kappa \gg 1$) is hard to undo.
It bounds how much relative errors can grow:
\begin{align}
  \frac{\|\delta \mathbf{x}\|}{\|\mathbf{x}\|}
  \lesssim \kappa(A)\,\frac{\|\delta \mathbf{b}\|}{\|\mathbf{b}\|}.
\end{align}

In [ ]:
for name, M in [("well-conditioned", A_good), ("ill-conditioned", A_bad)]:
    sigma = np.linalg.svd(M, compute_uv=False)  # ellipse semi-axes
    print(f"{name:17s}: sigma = {sigma},  kappa = {sigma[0]/sigma[-1]:8.2f},"
          f"  np.linalg.cond = {np.linalg.cond(M):8.2f}")

**Rule of thumb.**
Since floating-point input already has relative error
$\sim 10^{-16}$, the computed solution has relative error of up to
$\sim \kappa \times 10^{-16}$.
In other words, you lose about $\log_{10}\kappa$ of your 16 digits:

| $\kappa(A)$ | Digits you can trust (double precision) |
|---|---|
| $1$ to $10^2$ | 14 to 16: no worries |
| $10^6$ | about 10 |
| $10^{12}$ | about 4 |
| $\gtrsim 10^{16}$ | none: $A$ is numerically singular |

Note that if your *data* $\mathbf{b}$ have only 3 significant digits
(e.g., 0.1% photometry), even $\kappa = 10^3$ can wipe out all of the
useful information.

### How Well-Conditioned Is Our Spring Chain?

For the spring matrix one can show that
$\kappa(K) \approx 4(N+1)^2/\pi^2$, i.e., it grows like $N^2$.
This is typical of discretized differential equations: refining the
grid makes the matrix more ill-conditioned.

In [ ]:
for N in [10, 100, 1000]:
    print(f"N = {N:4d}:  kappa = {np.linalg.cond(spring_matrix(N)):.3e},"
          f"  estimate 4(N+1)^2/pi^2 = {4*(N+1)**2/np.pi**2:.3e}")

Even with $1000$ masses we only lose about 6 digits, so solving the
chain directly is safe.
The $N^2$ growth will come back when we discuss iterative methods:
it is also why they converge slowly.
Always check `np.linalg.cond()` when a solution looks suspicious.


## When $N$ Gets Large: Sparse Matrices and Iterative Solvers

### The $n^3$ Wall

Let's time `np.linalg.solve()` as the number of masses grows.

In [ ]:
def best_time(func, repeat=3):
    '''Shortest run time of func() over a few repeats, in seconds.'''
    times = []
    for _ in range(repeat):
        t0 = perf_counter()
        func()
        times.append(perf_counter() - t0)
    return min(times)

Ns      = np.array([100, 200, 400, 800, 1600, 3200])
t_dense = [best_time(lambda: np.linalg.solve(spring_matrix(N), np.ones(N))) for N in Ns]

plt.loglog(Ns, t_dense, 'o-', label='dense solve')
plt.loglog(Ns, t_dense[-1] * (Ns / Ns[-1])**3, 'k--', label=r'$\propto N^3$')
plt.xlabel('$N$'); plt.ylabel('time [sec]')
plt.legend()

(For small $N$ the timing is dominated by overheads; the $N^3$ trend
shows up at large $N$.)

Now think about a modest 3D gravity simulation on a $256^3$ grid.
That is $N \approx 1.7\times10^7$ unknowns.

In [ ]:
N = 256**3
print(f"N = {N:.2e} unknowns")
print(f"dense matrix memory: {8 * N**2 / 1e15:.0f} petabytes")
print(f"dense solve: ~{(2/3) * N**3 / 1e12 / 3600 / 24 / 365:.1e} years at 1 TFLOP/s")
# here TFLOP/s means tera floating point operations per second, a typical performance of a single multi-core computer

Clearly impossible.
The way out is to notice that the matrix is almost entirely zeros.

### Sparse Matrices

A matrix is **sparse** when most of its entries are zero.
Physical systems with local interactions (springs, grids, nearest
neighbors) always give sparse matrices.
`scipy.sparse` stores only the nonzero entries, so our tridiagonal
$K$ needs $3N$ numbers instead of $N^2$.

In [ ]:
def spring_matrix_sparse(N, k=1.0):
    '''Sparse version of spring_matrix().'''
    return k * sp.diags([-1.0, 2.0, -1.0], [-1, 0, 1], shape=(N, N), format='csr')

plt.spy(spring_matrix(20))
plt.title('nonzero entries of $K$ for $N=20$')

In [ ]:
t_sparse = [best_time(lambda: spla.spsolve(spring_matrix_sparse(N), np.ones(N))) for N in Ns]

plt.loglog(Ns, t_dense,  'o-', label='dense solve')
plt.loglog(Ns, t_sparse, 's-', label='sparse solve')
plt.xlabel('$N$'); plt.ylabel('time [sec]')
plt.legend()

N = 10**6
print(f"sparse solve with N = {N:.0e}: {best_time(lambda: spla.spsolve(spring_matrix_sparse(N), np.ones(N)), 1):.3f} sec")

For a 1D chain, a sparse *direct* solver is extremely efficient.
In 2D and 3D, however, elimination creates new nonzeros ("fill-in")
and direct methods again become too expensive.
That is where iterative methods come in.

## Iterative Solvers for Large/Sparse Systems

Direct methods are powerful but quickly become impractical when
dealing with very large or sparse systems.
In such cases, iterative solvers provide a scalable alternative.

* Large-Scale Problems:
  PDE discretizations in 2D/3D can easily lead to systems with
  millions of unknowns, making $\mathcal{O}(n^3)$ direct approaches
  infeasible.
* Sparse Matrices:
  Many physical systems produce matrices with a significant number of
  zero entries.
  Iterative methods may be implemented to access only nonzero elements
  each iteration, saving time and memory.
* Scalability:
  On parallel architectures (clusters, GPUs, etc), iterative methods
  often scale better than direct factorizations.

### Jacobi Iteration

1. Idea:
   Rewrite
   $A\mathbf{x} = \mathbf{b}$ as $\mathbf{x} = D^{-1}(\mathbf{b} - R\mathbf{x})$,
   where $D$ is the diagonal of $A$ and $R$ is the remainder.

2. Update Rule:
   \begin{align}
     x_i^{(k+1)} = \frac{1}{a_{ii}} \Big(b_i - \sum_{j \neq i} a_{ij} x_j^{(k)}\Big).
   \end{align}

3. Pros/Cons:
   * Easy to implement: each iteration updates $\mathbf{x}$ using only
     values from the previous iteration.
   * Slow convergence unless $A$ is well-conditioned (e.g., diagonally
     dominant).

In [ ]:
def solve_Jacobi(A, b, max_iter=100000, tol=1e-8):
    '''
    Solve A x = b using Jacobi iteration.
    Returns the solution, the relative residual at each iteration,
    and the history of x.
    '''
    D = np.diag(A)          # diagonal entries
    R = A - np.diag(D)      # off-diagonal part

    x = np.zeros_like(b)
    history, residuals = [x], []
    for k in range(max_iter):
        x = (b - R @ x) / D
        history.append(x)
        residuals.append(np.linalg.norm(b - A @ x) / np.linalg.norm(b))
        if residuals[-1] < tol:
            print(f"Jacobi converged in {k+1} iterations.")
            break
    else:
        print("Jacobi did not converge within max_iter.")
    return x, np.array(residuals), np.array(history)

In [ ]:
N = 30
K = spring_matrix(N)
f = 0.01 * np.ones(N)

x_jacobi, res_jacobi, hist = solve_Jacobi(K, f)
x_exact = np.linalg.solve(K, f)

i = np.arange(1, N+1)
for k in [1, 10, 100, 1000]:
    plt.plot(i, hist[k], label=f'iteration {k}')
plt.plot(i, x_exact, 'k--', label='exact')
plt.xlabel('mass index $i$'); plt.ylabel('displacement $x_i$')
plt.legend(loc='upper left')

**When does Jacobi converge?**
Each iteration multiplies the error by the matrix $D^{-1}R$.
Jacobi converges if all eigenvalues of $D^{-1}R$ have magnitude less
than 1, and the error shrinks by roughly the largest magnitude,
$\rho$ (the **spectral radius**), per iteration.
A sufficient condition is that $A$ is **diagonally dominant**
($|a_{ii}| \ge \sum_{j\neq i}|a_{ij}|$, with strict inequality in at least
one row).
For the spring chain $\rho = \cos(\pi/(N+1))$, which approaches 1 as $N$
grows, so the number of iterations grows like $N^2$, the same $N^2$ we saw in
the condition number.

In [ ]:
D = np.diag(np.diag(K))
rho = max(abs(np.linalg.eigvals(np.linalg.solve(D, K - D))))
print(f"spectral radius = {rho:.6f}, cos(pi/(N+1)) = {np.cos(np.pi/(N+1)):.6f}")
print(f"iterations to reduce error by 1e-8: ~{np.log(1e-8)/np.log(rho):.0f}")

### Gauss-Seidel: Use the Newest Information

Jacobi updates every mass using the *old* positions of its neighbors.
But when we update mass $i$, mass $i-1$ has *already* been updated in
this sweep.
Using that newest value is the **Gauss-Seidel** iteration:
\begin{align}
  x_i^{(k+1)} = \frac{1}{a_{ii}} \Big( b_i - \sum_{j < i} a_{ij}\, x_j^{(k+1)} - \sum_{j > i} a_{ij}\, x_j^{(k)} \Big).
\end{align}
It is just as cheap, needs no extra copy of $\mathbf{x}$, and
typically converges about twice as fast as Jacobi.

In [ ]:
# HANDSON: turn this loop-based Jacobi iteration into Gauss-Seidel.
#          Only ONE thing needs to change!

def solve_GS(A, b, max_iter=100000, tol=1e-8):
    '''
    Solve A x = b using Gauss-Seidel iteration.
    Returns the solution and the relative residual at each iteration.
    '''
    n = len(b)
    x = np.zeros_like(b)
    residuals = []
    for k in range(max_iter):
        x_old = x.copy()
        for i in range(n):
            # TODO: which of these should use the newest values in `x`?
            s = A[i, :i] @ x_old[:i] + A[i, i+1:] @ x_old[i+1:]
            x[i] = (b[i] - s) / A[i, i]
        residuals.append(np.linalg.norm(b - A @ x) / np.linalg.norm(b))
        if residuals[-1] < tol:
            print(f"Gauss-Seidel converged in {k+1} iterations.")
            break
    return x, np.array(residuals)

In [ ]:
# HANDSON: compare the convergence of Jacobi and Gauss-Seidel.

x_GS, res_GS = solve_GS(K, f)

plt.semilogy(res_jacobi, label='Jacobi')
plt.semilogy(res_GS,     label='Gauss-Seidel')
plt.xlabel('iteration'); plt.ylabel('relative residual')
plt.legend()

### What Practitioners Actually Use: Conjugate Gradient

Jacobi and Gauss-Seidel are great for building intuition, but they
are too slow to be used on their own.
For symmetric positive-definite matrices (which includes the spring
chain and the Poisson equation), the method of choice is
**conjugate gradient (CG)**.
Roughly speaking, instead of letting every mass relax locally, CG
chooses a smart search direction at each step that never undoes the
progress of previous steps.
In exact arithmetic it converges in at most $N$ iterations, and
usually much faster.

In [ ]:
res_CG = []
callback = lambda xk: res_CG.append(np.linalg.norm(f - K @ xk) / np.linalg.norm(f))
x_CG, info = spla.cg(spring_matrix_sparse(N), f, rtol=1e-8, callback=callback)

plt.semilogy(res_jacobi, label='Jacobi')
plt.semilogy(res_CG, 'o-', label='conjugate gradient')
plt.xlabel('iteration'); plt.ylabel('relative residual')
plt.xscale('log')
plt.legend()

print(f"Jacobi: {len(res_jacobi)} iterations;  CG: {len(res_CG)} iterations")

```{admonition} In the real world
:class: note
* `scipy.sparse.linalg` provides `cg` (symmetric positive-definite),
  `gmres` and `bicgstab` (general matrices), and `spsolve` (sparse
  direct).
* Iterative solvers are almost always combined with a
  **preconditioner**, an approximate inverse that makes the system
  better conditioned and cuts the number of iterations dramatically.
* For Poisson's equation on a grid, astrophysical simulation codes
  typically use **multigrid** methods (relax on a hierarchy of coarse
  and fine grids, so information travels across the domain quickly)
  or **FFT**-based solvers.
  Both fix exactly the "information travels one cell per iteration"
  problem we saw with Jacobi.
```

## Letting Go: Vibrations and Eigenvalues

So far the chain sat still.
Now remove the external forces, displace the masses, and let go.
Newton's second law for each mass gives
\begin{align}
  m\,\ddot{x}_i = k(x_{i-1} - x_i) + k(x_{i+1} - x_i)
  \quad\Longrightarrow\quad
  m\,\ddot{\mathbf{x}} = -K\mathbf{x},
\end{align}
with the *same* stiffness matrix $K$ as before.

### Normal Modes

Generic motions look complicated, but there are special patterns,
called **normal modes**, in which every mass oscillates at the same
frequency and the *shape* of the chain stays the same:
\begin{align}
  \mathbf{x}(t) = \mathbf{v}e^{\,i\,\omega\,t}.
\end{align}
Plugging this in gives
\begin{align}
  K\mathbf{v} = m\omega^2\,\mathbf{v}.
\end{align}
This is an **eigenvalue problem** $A\mathbf{v} = \lambda\mathbf{v}$:
find the special vectors $\mathbf{v}$ (eigenvectors) that $A$ only
stretches, by a factor $\lambda$ (eigenvalue), without changing
their direction.
Here each eigenvector is a mode shape and each eigenvalue gives a
mode frequency, $\omega = \sqrt{\lambda/m}$.

### Back to the Picture

For $N=2$ masses, $K = k\begin{pmatrix}2 & -1\\ -1 & 2\end{pmatrix}$.
Let's draw what it does to the unit circle, together with its
eigenvectors.

In [ ]:
K2 = spring_matrix(2)
lam, V = np.linalg.eigh(K2)
V *= np.sign(V[0])  # eigenvectors are only defined up to a sign

fig, ax = plt.subplots(figsize=(5, 5))
theta  = np.linspace(0, 2*np.pi, 200)
circle = np.array([np.cos(theta), np.sin(theta)])
ax.plot(*circle, 'k--', lw=1, label='unit circle')
ax.plot(*(K2 @ circle), 'C0', lw=2, label='after $K$')
for j, c in enumerate(['C1', 'C2']):
    v = V[:, j]
    ax.arrow(0, 0, *v,          color=c, width=0.02, length_includes_head=True, alpha=0.4)
    ax.arrow(0, 0, *(K2 @ v),   color=c, width=0.04, length_includes_head=True,
             label=rf'$K\mathbf{{v}}_{j+1} = {lam[j]:.0f}\,\mathbf{{v}}_{j+1}$')
ax.set_aspect('equal')
ax.legend(loc='upper left')

print("eigenvalues :", lam)
print("eigenvectors (columns):\n", V)

The eigenvectors lie exactly along the axes of the ellipse: $K$
stretches them without rotating them.
They also have a clear physical meaning:
* $\mathbf{v}_1 \propto (1, 1)$, $\omega_1 = \sqrt{k/m}$: both masses
  move **together** (in phase); the middle spring is never stretched.
* $\mathbf{v}_2 \propto (1, -1)$, $\omega_2 = \sqrt{3k/m}$: the masses
  move **opposite** each other (out of phase); the middle spring
  works hard, so the frequency is higher.

You can verify these by hand by solving $\det(K - m\omega^2 I) = 0$.

:::{admonition} Background: why `eigh()` and not `eig()`?
:class: tip
$K$ is **symmetric** ($K = K^\top$; physically, Newton's third law:
the force of mass $i$ on $j$ equals that of $j$ on $i$).
Symmetric matrices are special:
* their eigenvalues are always real,
* their eigenvectors are orthogonal to each other,
* and they can be computed faster and more accurately.

`np.linalg.eigh()` exploits this and returns eigenvalues sorted from
smallest to largest.
Use `np.linalg.eig()` only for non-symmetric matrices; its results
may be complex and are not sorted.
:::

### Many Masses: Standing Waves

Now take $N = 10$ masses and look at the mode shapes.

In [ ]:
m, k = 1.0, 1.0
N = 10
lam, V = np.linalg.eigh(spring_matrix(N, k))
omega  = np.sqrt(lam / m)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

i = np.arange(N+2)  # include the walls at i = 0 and N+1
for j in range(3):
    v = np.concatenate([[0], V[:, j], [0]])
    v *= np.sign(v[1])  # eigenvectors are only defined up to a sign
    axes[0].plot(i, v, 'o-', label=rf'mode {j+1}, $\omega$ = {omega[j]:.3f}')
axes[0].set_xlabel('mass index $i$'); axes[0].set_ylabel('mode shape')
axes[0].legend()

j = np.arange(1, N+1)
axes[1].plot(j, omega, 'o', label='numerical (eigh)')
axes[1].plot(j, 2*np.sqrt(k/m) * np.sin(j*np.pi / (2*(N+1))), 'k--', label='analytic')
axes[1].set_xlabel('mode number $j$'); axes[1].set_ylabel(r'frequency $\omega_j$')
axes[1].legend()
fig.tight_layout()

The modes are standing waves: mode $j$ has $j-1$ nodes, and higher
modes oscillate faster, exactly like a guitar string.
The numerical frequencies match the analytic result
$\omega_j = 2\sqrt{k/m}\,\sin\!\big(j\pi/2(N+1)\big)$.

```{admonition} Astronomy connection: asteroseismology
:class: note
Stars ring too.
The Sun oscillates in millions of normal modes, and space missions such as Kepler and TESS measure mode
frequencies in thousands of other stars.
Because each mode samples the stellar interior differently, the
frequencies constrain the star's internal structure, rotation, and
age.
Computing the modes of a stellar model is, at its core, the same
eigenvalue problem as our chain, just with a far more complicated
"stiffness" matrix.
(The same mathematics appears in quantum mechanics, where
$\hat{H}|E\rangle = E|E\rangle$ gives energy levels.)
```

### Putting the Modes Together

Because the eigenvectors of a symmetric matrix are orthogonal, *any*
initial displacement $\mathbf{x}(0)$ can be written as a sum of modes
with coefficients $c_j = \mathbf{v}_j \cdot \mathbf{x}(0)$
(the coefficient of each eigenvector equals the inner product).
Each mode then simply oscillates at its own frequency.
Starting from rest,
\begin{align}
  \mathbf{x}(t) = \sum_j c_j\,\mathbf{v}_j \cos(\omega_j t)
  = V\,\big[\mathbf{c} \odot \cos(\boldsymbol{\omega} t)\big],
  \qquad \mathbf{c} = V^\top\mathbf{x}(0),
\end{align}
where $V$ has the eigenvectors as columns and $\odot$ is element-wise
multiplication.
The eigenvalue problem has turned $N$ coupled equations into $N$
independent harmonic oscillators.

In [ ]:
# HANDSON: find solution to an initial value problem

def chain_motion(K, x0, m=1.0):
    '''Return a function x(t) for a chain released from rest at x0.'''
    ...

Check against the analytic two-mass solution, released with
$x_1(0) = 0$, $x_2(0) = 0.5$:
\begin{align}
  x_{1,2}(t) = \frac{x_1(0)+x_2(0)}{2}\cos\omega_1 t \pm \frac{x_1(0)-x_2(0)}{2}\cos\omega_2 t.
\end{align}

In [ ]:
x0 = np.array([0.0, 0.5])
X  = chain_motion(spring_matrix(2), x0)

def X_analytic(t, x0=x0, m=1.0, k=1.0):
    xp = (x0[0] + x0[1]) / 2 * np.cos(np.sqrt(k/m) * t)
    xm = (x0[0] - x0[1]) / 2 * np.cos(np.sqrt(3*k/m) * t)
    return np.array([xp + xm, xp - xm])

for t in [0.0, 1.0, 5.0]:
    print(f"t = {t}: eigen-solution {X(t)}, analytic {X_analytic(t)}")

Now pull the first mass of a 20-mass chain and let go.
A "space-time" diagram below shows the motion of every mass at once.

In [ ]:
N  = 20
x0 = np.zeros(N); x0[0] = 0.5
X  = chain_motion(spring_matrix(N), x0)

T  = np.linspace(0, 60, 600)
XT = np.array([X(t) for t in T])

plt.imshow(XT.T, aspect='auto', origin='lower', cmap='RdBu_r',
           extent=[T[0], T[-1], 0.5, N+0.5], vmin=-0.3, vmax=0.3)
plt.colorbar(label='displacement')
plt.xlabel('time $t$'); plt.ylabel('mass index $i$')

The pulse travels down the chain, reflects off the far wall, and
comes back: a wave, built entirely out of standing-wave modes.
We can also watch it as a movie.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

def mkanim(X, T, N):
    '''Animate a chain of N masses whose displacements are X(t).'''
    fig, ax = plt.subplots(figsize=(10, 1.5))
    ax.set_xlim(-0.5, N+1.5); ax.set_ylim(-0.5, 0.5); ax.set_yticks([])
    ax.axvline(0,   color='k', lw=4)
    ax.axvline(N+1, color='k', lw=4)
    springs, = ax.plot([], [], '-',  color='gray', lw=1)
    masses,  = ax.plot([], [], 'o',  color='C0',   ms=10)

    def animate(n):
        pos = np.arange(1, N+1) + X(T[n])
        springs.set_data(np.concatenate([[0], pos, [N+1]]), np.zeros(N+2))
        masses.set_data(pos, np.zeros(N))
        return springs, masses

    anim = animation.FuncAnimation(fig, animate, frames=len(T), interval=40, blit=True)
    plt.close(fig)
    return anim

HTML(mkanim(X, np.linspace(0, 40, 200), N).to_jshtml())

In [ ]:
# HANDSON: Release the 20-mass chain in a single normal mode

### How Do Computers Find Eigenvalues? The Power Method

Here is the simplest idea, and it is surprisingly useful.
Write any starting vector as a sum of eigenvectors,
$\mathbf{x} = \sum_j c_j \mathbf{v}_j$.
Multiplying by $A$ repeatedly gives
\begin{align}
  A^k\mathbf{x} = \sum_j c_j \lambda_j^k\,\mathbf{v}_j.
\end{align}
The term with the largest $|\lambda_j|$ grows fastest and eventually
dominates: **keep applying $A$ (and renormalizing), and the vector
turns toward the dominant eigenvector.**
The other components die off like $|\lambda_2/\lambda_1|^k$, so
convergence is fast when the top eigenvalue is well separated, and
slow when it is not.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])

x = np.array([1.0, -0.2])
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(np.cos(np.linspace(0, 2*np.pi, 200)), np.sin(np.linspace(0, 2*np.pi, 200)), 'k--', lw=1)
for k in range(6):
    x = x / np.linalg.norm(x)
    ax.arrow(0, 0, *x, color=plt.cm.viridis(k/5), width=0.02, length_includes_head=True)
    ax.annotate(str(k), 1.1*x)
    x = A @ x

lam, V = np.linalg.eigh(A)
ax.plot(*np.array([-V[:, -1], V[:, -1]]).T, 'r-', lw=1, label='dominant eigenvector')
ax.set_aspect('equal'); ax.legend(loc='lower left')

In [ ]:
def eig_power(A, max_iter=100000, tol=1e-10):
    '''
    Returns the eigenvalue of largest magnitude and its eigenvector.
    '''
    # Start from a random vector: a "nice" guess such as all ones can be
    # exactly orthogonal to the dominant eigenvector (e.g., by symmetry).
    v   = np.random.default_rng(0).standard_normal(A.shape[0])
    v  /= np.linalg.norm(v)
    lam = v @ A @ v
    for k in range(max_iter):
        w       = A @ v
        v       = w / np.linalg.norm(w)
        lam_old = lam
        lam     = v @ A @ v  # Rayleigh quotient
        if abs(lam - lam_old) < tol * abs(lam):
            print(f"Power method converged in {k+1} iterations.")
            return lam, v
    print("Power method did not converge within max_iter.")
    return lam, v

lam_dom, v_dom = eig_power(A)
print(lam_dom, v_dom, "power method")
print(np.linalg.eigh(A)[0][-1], np.linalg.eigh(A)[1][:, -1], "numpy")

For the spring chain, the power method finds the *highest*-frequency
mode (all neighbors moving in opposite directions).
Physically we usually care more about the *lowest* modes.
The trick is to apply the power method to $K^{-1}$, whose largest
eigenvalue is $1/\lambda_{\min}$.
And of course we never form $K^{-1}$; we *solve* with an $LU$
factorization computed once.
This is called **inverse iteration**.

```{admonition} What libraries actually do
:class: note
`np.linalg.eigh()` and `np.linalg.eig()` call LAPACK routines based
on the **QR algorithm**, a much more sophisticated relative of the
power method that finds *all* eigenpairs of a dense matrix in
$\mathcal{O}(n^3)$ operations.
For large sparse matrices where you only need a few modes, use
`scipy.sparse.linalg.eigsh()`, which builds on power-method-like
(Lanczos) iterations.
With `sigma=0` it finds the eigenvalues closest to zero, using
inverse iteration under the hood.
```

In [ ]:
N = 100000
lam, V = spla.eigsh(spring_matrix_sparse(N), k=4, sigma=0)
print("lowest 4 eigenvalues:", [f"{l:.6e}" for l in lam])
print("analytic            :", [f"{l:.6e}" for l in 4*np.sin(np.arange(1, 5)*np.pi / (2*(N+1)))**2])

## Summary

* Physical problems become linear algebra: forces in equilibrium give
  $A\mathbf{x}=\mathbf{b}$; vibrations give $A\mathbf{v}=\lambda\mathbf{v}$.
  The same matrix can describe springs, gravity on a grid, or waves.
* A matrix stretches space; the shape of that stretch controls both
  how hard $A\mathbf{x}=\mathbf{b}$ is (condition number) and what the
  natural modes are (eigenvectors).
* **Stability** is a property of the algorithm (fix it with pivoting);
  **conditioning** is a property of the problem (you lose about
  $\log_{10}\kappa$ digits; fix it by reformulating).
* Dense direct solvers cost $\mathcal{O}(n^3)$.
  Large physical problems are sparse; use sparse storage and sparse
  direct or iterative (CG + preconditioner, multigrid) solvers.
* Eigenvectors decouple a system into independent modes;
  use `eigh` for symmetric matrices and `eigsh` for a few modes of a
  large sparse one.

## Looking Ahead

### Singular Value Decomposition (SVD)

The ellipse picture generalizes to any matrix, even non-square ones:
[SVD](https://en.wikipedia.org/wiki/Singular_value_decomposition)
factorizes $A = U \Sigma V^\top$, where the singular values in
$\Sigma$ are the ellipse's semi-axes.
It gives the condition number, the most stable least-squares
solutions, pseudo-inverses, and optimal low-rank approximations
(compression and noise reduction).

### Principal Component Analysis (PCA)

[PCA](https://en.wikipedia.org/wiki/Principal_component_analysis)
applies the SVD to a data set to find the directions of maximum
variance.
In astronomy, PCA is used to compress large samples of galaxy and
quasar spectra into a handful of "eigenspectra", to remove
systematics from light curves, and to model point spread functions.